In [36]:
# ===============================
# Cell 0 — Imports & utilities
# ===============================

import pandas as pd
import numpy as np
import json
import os
import re
from tqdm import tqdm

from sklearn.model_selection import train_test_split

from openai import OpenAI


In [37]:
# Load API key
with open("key.txt", "r") as f:
    api_key = f.read().strip()

client = OpenAI(api_key=api_key)


In [38]:
# ===============================
# Cell 1 — Arabic text normalization
# ===============================

# Arabic tashkeel unicode range
TASHKEEL_REGEX = re.compile(r'[\u0617-\u061A\u064B-\u0652]')

def remove_tashkeel(text):
    if not isinstance(text, str):
        return ""
    return re.sub(TASHKEEL_REGEX, '', text).strip()


In [39]:
# ===============================
# Cell 2 — Load datasets
# ===============================

POS_PATH = "majaz_dataset.csv"
NEG_PATH = "non_majaz_random_sample.csv"

TEXT_COL = "البيت_الكامل"

df_pos = pd.read_csv(POS_PATH)
df_neg = pd.read_csv(NEG_PATH)

df_pos["label"] = 1
df_neg["label"] = 0

print("Majaz samples:", len(df_pos))
print("Non-Majaz samples:", len(df_neg))


Majaz samples: 842
Non-Majaz samples: 842


In [40]:
# ===============================
# Cell 3 — Merge & clean
# ===============================

df = pd.concat([df_pos, df_neg], ignore_index=True)
df = df[[TEXT_COL, "label"]].dropna()

# Normalize Arabic
df[TEXT_COL] = df[TEXT_COL].apply(remove_tashkeel)

df = df.reset_index(drop=True)

print("Final dataset size:", df.shape)
df.head()


Final dataset size: (1594, 2)


,البيت_الكامل,label
0,مررت على دار الحبيب فحمحمت جوادي وهل تشجو الجي...,1
1,وتسعدني في غمرة بعد غمرة سبوح لها منها عليها ش...,1
2,محرمة أكفال خيلي على القنا محللة لباتها والقلائد,1
3,ولكن إذا لم يحمل القلب كفه على حالة لم يحمل ال...,1
4,له من كريم الطبع في الحرب منتض ومن عادة الإحسا...,1


In [41]:
# ===============================
# Cell 4 — Train / Test split
# ===============================

X = df[TEXT_COL].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.05,
    stratify=y,
    random_state=42
)

print("Train:", len(X_train))
print("Test :", len(X_test))


Train: 1514
Test : 80


In [42]:
# ===============================
# GPT Translation (Arabic → English)
# ===============================

TRANSLATE_SYSTEM_PROMPT = """
You are a professional translator specialized in Classical Arabic poetry.

Rules:
- Translate the verse into natural, fluent English
- Preserve figurative meaning and imagery
- Do NOT explain
- Do NOT paraphrase excessively
- Output ONLY the English translation
"""

def translate_verse_gpt(arabic_verse):
    response = client.responses.create(
        model="gpt-4o-2024-08-06",
        input=[
            {"role": "system", "content": TRANSLATE_SYSTEM_PROMPT},
            {"role": "user", "content": arabic_verse}
        ]
    )
    return response.output_text.strip()


In [43]:
# ===============================
# Translate X_test to English
# ===============================

X_test_en = []

for verse in tqdm(X_test, desc="Translating verses"):
    try:
        en = translate_verse_gpt(verse)
    except Exception as e:
        print("Translation error:", e)
        en = ""
    X_test_en.append(en)

X_test_en[:5]


Translating verses: 100%|██████████| 80/80 [01:43<00:00,  1.30s/it]


['Her garment stained after the cold’s grace, sullied like the crumbling dune of soft sand.',
 'Gazelle-like steeds roam over Iraq, gathering the bounty of Nabateans.',
 'In every man there is an extra hand, and it carves upon his mice.',
 'To the sons of Abdul Karim, your eyes wandered—woe to you—whom do you surpass?',
 'Send a messenger to summon us a living sleep when you know they have grown heedless.']

In [44]:
# ===============================
# English MAJAZ classifier prompt
# ===============================

EN_MAJAZ_SYSTEM_PROMPT = """
You are an expert in Arabic rhetoric (Balaghah), specializing in MAJAZ.

You will receive an ENGLISH translation of an Arabic poetic verse.

Your task:
Decide whether the ORIGINAL Arabic verse contains MAJAZ,
based on the translated meaning.

Definition:
MAJAZ = non-literal semantic transfer supported by context.

Count as MAJAZ:
- Metaphor
- Metonymy
- Personification
- Attribution of actions to non-agents

Do NOT count:
- Literal language
- Explicit similes ("like", "as")
- Emotional exaggeration without semantic transfer

Output rules:
- Output ONLY one digit
- 1 = contains majaz
- 0 = does NOT contain majaz
- No explanation
"""

def classify_majaz_english_batch(english_verses):
    verses_block = "\n".join(
        [f"{i+1}. {v}" for i, v in enumerate(english_verses)]
    )

    prompt = f"""
For EACH verse below, decide if it contains MAJAZ.

Output:
- One number per line
- Same order
- No text

Verses:
{verses_block}
"""

    response = client.responses.create(
        model="gpt-4o-2024-08-06",
        input=[
            {"role": "system", "content": EN_MAJAZ_SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )

    lines = response.output_text.strip().splitlines()
    preds = [1 if l.strip().startswith("1") else 0 for l in lines]

    if len(preds) != len(english_verses):
        raise ValueError("Prediction length mismatch")

    return preds


In [45]:
# ===============================
# Predict using English translations
# ===============================

y_pred_en = []
BATCH_SIZE = 10
for i in tqdm(range(0, len(X_test_en), BATCH_SIZE), desc="Classifying (EN)"):
    batch = X_test_en[i:i+BATCH_SIZE]

    try:
        preds = classify_majaz_english_batch(batch)
        y_pred_en.extend(preds)
    except Exception as e:
        print("Batch error:", e)
        y_pred_en.extend([0] * len(batch))


Classifying (EN):  50%|█████     | 4/8 [00:04<00:04,  1.19s/it]

Batch error: Prediction length mismatch


Classifying (EN): 100%|██████████| 8/8 [00:09<00:00,  1.13s/it]


In [46]:
# ===============================
# Evaluation (English-based)
# ===============================
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

acc_en = accuracy_score(y_test, y_pred_en)
cm_en = confusion_matrix(y_test, y_pred_en)

print("Accuracy (English pipeline):", acc_en)
print("\nConfusion Matrix:")
print(cm_en)

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_en,
    target_names=["Non-Majaz", "Majaz"]
))


Accuracy (English pipeline): 0.575

Confusion Matrix:
[[20 18]
 [16 26]]

Classification Report:
              precision    recall  f1-score   support

   Non-Majaz       0.56      0.53      0.54        38
       Majaz       0.59      0.62      0.60        42

    accuracy                           0.57        80
   macro avg       0.57      0.57      0.57        80
weighted avg       0.57      0.57      0.57        80



In [47]:
# ===============================
# Save detailed results
# ===============================

results_en = pd.DataFrame({
    "verse_ar": X_test,
    "verse_en": X_test_en,
    "y_true": y_test,
    "y_pred_en": y_pred_en
})

OUT_PATH = "gpt_majaz_predictions_english_pipeline.csv"
results_en.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_PATH}")
results_en.head()


Saved to: gpt_majaz_predictions_english_pipeline.csv


,verse_ar,verse_en,y_true,y_pred_en
0,تلوث بعد افتضال البرد مئزرها، لوثا ، على مثل د...,"Her garment stained after the cold’s grace, su...",0,1
1,وخيل غزالة تنتابهم تجوب العراق وتحوي النبيطا,"Gazelle-like steeds roam over Iraq, gathering ...",1,1
2,في كل رجل ويد زائدة وهو على جرذانه قد شطبا,"In every man there is an extra hand, and it ca...",0,0
3,أإلى بني عبد الكريم تشاوست عيناك ويلك خلف من ت...,"To the sons of Abdul Karim, your eyes wandered...",0,0
4,دست رسولا أن ائتنا رقدة الـ ـحي إذا ما علمتهم ...,Send a messenger to summon us a living sleep w...,1,1
